In [88]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [115]:
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm
from scipy.stats import halfnorm

In [130]:
from simulators import NestedModelFamily
from simulators.benchmarks import DDM
from adapters import Adapter
from diagnostics.plot import recovery

In [117]:
from networks.transformers.gpt import BayesGPTv2, BayesGPTv1
from networks.loss import mse_loss

In [118]:
# ddm_priors = {
#     "v":        {"intercept": lambda: np.random.gamma(2., 1.),
#                  "slope": lambda: 0.0},
#     "a":        {"intercept": lambda: np.random.normal(-1, 0.3),
#                  "slope": lambda: 0.0},
#     "tau":      {"intercept": lambda: np.random.normal(-1.5, 0.3),
#                  "slope": lambda: 0.0},
#     "s_v":      {"intercept": lambda: halfnorm.rvs(loc=0.0, scale=1.0),
#                  "slope": lambda: 0.0},
#     "s_tau":    {"intercept": lambda: np.random.beta(1.0, 3.0),
#                  "slope": lambda: 0.0}
# }

In [131]:
ddm_full_priors = {
    "v":        {"intercept": lambda: np.random.gamma(2., 1.),
                 "slope": lambda: np.random.normal(0., 1.)},
    "a":        {"intercept": lambda: np.random.normal(-1, 0.3),
                 "slope": lambda: np.random.normal(0., 1.)},
    "tau":      {"intercept": lambda: np.random.normal(-1.5, 0.3),
                 "slope": lambda: np.random.normal(0., 1.)},
    "s_v":      {"intercept": lambda: halfnorm.rvs(loc=0.0, scale=1.0),
                 "slope": lambda: np.random.normal(0., 1.)},
    "s_tau":    {"intercept": lambda: np.random.beta(1.0, 3.0),
                 "slope": lambda: np.random.normal(0., 1.)}
}

In [132]:
model_family = NestedModelFamily(name="DDM", model=DDM(), prior_fun=ddm_full_priors)

In [133]:
samples = model_family.batch_sample(
    batch_size=1,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "s_tau"},
        fixed_intrinsics={}
    ),
    num_obs=500,
    flatten_param_outputs=True
)

In [134]:
adapter = Adapter()

### BayesGPT

In [135]:
bayesgpt = BayesGPTv1(encoder_num_layers=8, decoder_num_layers=8, seed_dim=128, num_seeds=40)

In [136]:
grad_clip_norm = 5.
batch_size = 32
epochs = 100
steps_per_epoch = 100
learning_rate = 2e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bayesgpt.to(device)
bayesgpt.train()#### HByp

optimizer = Adam(bayesgpt.parameters(), lr=learning_rate)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

for ep in range(epochs):
    pbar = tqdm(total=steps_per_epoch, desc=f"Epoch {ep+1}/{epochs}", miniters=100)

    for step in range(steps_per_epoch):

        samples = model_family.batch_sample(
            batch_size=batch_size,
            mask_randomizer_kwargs=dict(
                free_intrinsics={"v", "a", "tau", "s_v", "s_tau"},
                fixed_intrinsics={}
            ),
            num_obs=500,
            flatten_param_outputs=True
        )

        adapted = adapter.adapt(samples, intrinsic_params=model_family.intrinsic_params)

        optimizer.zero_grad()

        mu, logvar = bayesgpt(
            adapted["input_data"],
            adapted["param_indices"],
            adapted["regressor_indices"],
            adapted["param_masks"]
        )

        L = mse_loss(adapted["param_matrices"], mu, logvar, adapted["param_masks"])
        L.backward()

        if grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(bayesgpt.parameters(), grad_clip_norm)

        optimizer.step()

        loss_val = L.detach().item()
        current_lr = scheduler.get_last_lr()[0]

        pbar.set_postfix(loss=f"{loss_val:.4f}", lr=f"{current_lr:.2e}")
        pbar.update(1)

    scheduler.step()
    pbar.close()

Epoch 1/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 2/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 3/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 4/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 5/100:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch 96/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 97/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 98/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 99/100:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch 100/100:   0%|          | 0/100 [00:00<?, ?it/s]